In [0]:
from datetime import datetime
import pandas as pd

In [0]:
%sql
WITH events_mobile AS (
SELECT 
  date(e_derived_tstamp) AS e_derived_tstamp_date
  , cssc_session_id as session_id
  --, e_event_id
  --, e_app_id
  , bcpu_user_id
  , TRIM(e_geo_region) AS e_geo_region
  , TRIM(e_geo_city) AS e_geo_city
  --, e_geo_region_name
  --, bcpu_user_specialty
  , bcpt_topic_id
FROM analytics_spm_prod.analytics_b2p_whitebook_curated.whitebook_snowplow_mobile_view_curated
WHERE bcpv_view_type = 'open'
  AND e_derived_tstamp >= '2021-01-01'
  AND e_derived_tstamp < current_date
  AND e_app_id in ('whitebook_android', 'whitebook_ios', 'crackedVersion') 
  AND bcpu_user_formation_degree = 'profissional'
  AND e_geo_country = 'BR'
  AND bcpu_user_occupation_area = 1 -- Médico
  ),
events_web as (
SELECT 
  date(e_derived_tstamp) AS e_derived_tstamp_date
  , e_domain_sessionid as session_id
  --, e_event_id
  --, e_app_id
  , bcpu_user_id
  , TRIM(e_geo_region) AS e_geo_region
  , TRIM(e_geo_city) AS e_geo_city
  --, e_geo_region_name
  --, bcpu_user_specialty
  , bcpt_topic_id
FROM analytics_spm_prod.analytics_b2p_whitebook_curated.whitebook_snowplow_web_view_curated
WHERE bcpv_type = 'open' 
  AND e_derived_tstamp >= '2021-01-01'
  AND e_derived_tstamp < current_date
  AND e_app_id in ('whitebook_web', 'wbweb') 
  AND bcpu_user_formation_degree = 'profissional'
  AND e_geo_country = 'BR'
  AND bcpu_user_occupation_area = 1 -- Médico
),
union_events as (
SELECT * FROM events_mobile
union
SELECT * FROM events_web
),
main_theme AS (
SELECT DISTINCT
  cm.content_id,
  tm.value,
  cm.kafka_processed_at
FROM bronze_source_prod.analytics_whitebook_pebmedapps_source.whitebook_pebmedapps_conteudo_metadado_source cm 
LEFT JOIN bronze_source_prod.analytics_whitebook_pebmedapps_source.whitebook_pebmedapps_tipo_metadado_source tm 
ON cm.id_metadata_type = tm.id
WHERE type = 'TEMA' AND level = 'PRINCIPAL'
),
maxdate_main_theme AS (
SELECT 
  content_id,
  MAX(kafka_processed_at) AS max_kafka_processed_at
FROM main_theme
GROUP BY 1
),
select_maxdate_main_theme AS (
SELECT 
  mt.content_id,
  mt.value AS content_main_theme,
  max_kafka_processed_at
FROM main_theme mt
INNER JOIN maxdate_main_theme mmt
ON (mt.content_id = mmt.content_id AND mt.kafka_processed_at = mmt.max_kafka_processed_at)
),
type_theme AS (
SELECT DISTINCT
  cm.content_id,
  tm.value AS content_type_theme
FROM bronze_source_prod.analytics_whitebook_pebmedapps_source.whitebook_pebmedapps_conteudo_metadado_source cm 
LEFT JOIN bronze_source_prod.analytics_whitebook_pebmedapps_source.whitebook_pebmedapps_tipo_metadado_source tm 
	ON cm.id_metadata_type = tm.id
WHERE type = 'TIPO_TEMA'
),
union_events_theme AS (
SELECT *
FROM union_events ue
LEFT JOIN select_maxdate_main_theme smmt
  ON ue.bcpt_topic_id = smmt.content_id
LEFT JOIN type_theme tt
  ON ue.bcpt_topic_id = tt.content_id
),
city_regex AS (
SELECT *
, REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(UPPER(e_geo_city), 'Ã', 'A'), 'Á', 'A'), 'À', 'A'), 'Â', 'A'), 'É', 'E'), 'È', 'E'), 'Ê', 'E'), 'Í', 'I'), 'Ì', 'I'), 'Õ', 'O'), 'Ô', 'O'), 'Ó', 'O'), 'Ò', 'O'), 'Ú', 'U'), 'Ù', 'U') AS upper_e_geo_city
FROM union_events_theme
),
index_rc AS (
SELECT *
	, ROW_NUMBER() OVER (PARTITION BY session_id, content_main_theme ORDER BY e_derived_tstamp_date) AS rc_session_index
FROM city_regex
)
SELECT
  e_derived_tstamp_date as access_date 
  , content_main_theme
  , e_geo_region as access_state
  , upper_e_geo_city AS access_city
  , COUNT(*)
FROM index_rc
WHERE content_main_theme IS NOT NULL
	--AND content_type_theme IN ('Doença', 'Síndrome', 'Sinal/Sintoma')
    AND e_geo_region IS NOT NULL
    AND content_main_theme IN ('Dengue')
    AND upper_e_geo_city IN ('PADRE PARAISO',
    'PAIVA',
    'PASSA TEMPO',
    'PASSA VINTE',
    'PASSOS',
    'PAVAO',
    'PEDRO LEOPOLDO',
    'PEDRO TEIXEIRA',
    'PESCADOR',
    'PIAU',
    'PIEDADE DE CARATINGA',
    'PIEDADE DOS GERAIS',
    'PIRACEMA',
    'PIRAPORA',
    'PONTO CHIQUE',
    'PONTO DOS VOLANTES',
    'POTE',
    'PRATA',
    'PRATAPOLIS',
    'RAPOSOS',
    'RAUL SOARES',
    'RIBEIRAO DAS NEVES',
    'RIO ACIMA',
    'RIO MANSO',
    'RIO NOVO',
    'RIO PRETO',
    'SABARA',
    'SACRAMENTO',
    'SANTA BARBARA DO LESTE',
    'SANTA BARBARA DO MONTE VERDE',
    'SANTA FE DE MINAS',
    'SANTA JULIANA',
    'SANTA LUZIA',
    'SANTA RITA DE JACUTINGA',
    'SANTA RITA DE MINAS',
    'SANTANA DO DESERTO',
    'SANTOS DUMONT',
    'SAO DOMINGOS DAS DORES',
    'SAO FELIX DE MINAS',
    'SAO FRANCISCO DE PAULA',
    'SAO JOAO BATISTA DO GLORIA',
    'SAO JOAO DO MANTENINHA',
    'SAO JOAQUIM DE BICAS',
    'SAO JOSE DA BARRA',
    'SAO JOSE DA LAPA',
    'SAO JOSE DO DIVINO',
    'SAO SEBASTIAO DO ANTA',
    'SARZEDO',
    'SERRA DOS AIMORES',
    'SETUBINHA',
    'SIMAO PEREIRA',
    'TAQUARACU DE MINAS',
    'TEOFILO OTONI',
    'TUPACIGUARA',
    'UBAPORANGA',
    'UBERABA',
    'UBERLANDIA',
    'VARGEM ALEGRE',
    'VARZEA DA PALMA',
    'VERISSIMO',
    'VERMELHO NOVO',
    'VESPASIANO',
    'AFONSO CLAUDIO',
    'ALEGRE',
    'APIACA',
    'ARACRUZ',
    'BOM JESUS DO NORTE',
    'BREJETUBA',
    'CONCEICAO DO CASTELO',
    'DIVINO DE SAO LOURENCO',
    'DOMINGOS MARTINS',
    'DORES DO RIO PRETO',
    'GUACUI',
    'IBATIBA',
    'IBIRACU',
    'IBITIRAMA',
    'IRUPI',
    'ITARANA',
    'IUNA',
    'JOAO NEIVA',
    'LARANJA DA TERRA',
    'LINHARES',
    'MARECHAL FLORIANO',
    'MUNIZ FREIRE',
    'RIO BANANAL',
    'SANTA LEOPOLDINA',
    'SANTA MARIA DE JETIBA',
    'SANTA TERESA',
    'SAO JOSE DO CALCADO',
    'SOORETAMA',
    'VENDA NOVA DO IMIGRANTE',
    'BARRA DO PIRAI',
    'BARRA MANSA',
    'BELFORD ROXO',
    'CAMPOS DOS GOYTACAZES',
    'CARDOSO MOREIRA',
    'DUQUE DE CAXIAS',
    'ENGENHEIRO PAULO DE FRONTIN',
    'GUAPIMIRIM',
    'ITABORAI',
    'ITAGUAI',
    'ITALVA',
    'ITATIAIA',
    'JAPERI',
    'MAGE',
    'MANGARATIBA',
    'MARICA',
    'MENDES',
    'MESQUITA',
    'NILOPOLIS',
    'NITEROI',
    'NOVA IGUACU',
    'PARACAMBI',
    'PINHEIRAL',
    'PIRAI',
    'PORTO REAL',
    'QUATIS',
    'QUEIMADOS',
    'RESENDE',
    'RIO CLARO',
    'RIO DE JANEIRO',
    'SAO FIDELIS',
    'SAO FRANCISCO DE ITABAPOANA',
    'SAO GONCALO',
    'SAO JOAO DA BARRA',
    'SAO JOAO DE MERITI',
    'SAQUAREMA',
    'SEROPEDICA',
    'TANGUA',
    'VOLTA REDONDA',
    'ALTINOPOLIS',
    'ALVARO DE CARVALHO',
    'ALVINLANDIA',
    'ARUJA',
    'BARRINHA',
    'BARUERI',
    'BATATAIS',
    'BIRITIBA MIRIM',
    'BRODOWSKI',
    'CAIEIRAS',
    'CAJAMAR',
    'CAJURU',
    'CAMPOS NOVOS PAULISTA',
    'CARAPICUIBA',
    'CASSIA DOS COQUEIROS',
    'COTIA',
    'CRAVINHOS',
    'DIADEMA',
    'DUMONT',
    'ECHAPORA',
    'EMBU DAS ARTES',
    'EMBUGUACU',
    'FERNAO',
    'FERRAZ DE VASCONCELOS',
    'FRANCISCO MORATO',
    'FRANCO DA ROCHA',
    'GALIA',
    'GARCA',
    'GETULINA',
    'GUAIMBE',
    'GUARAREMA',
    'GUARIBA',
    'GUARULHOS',
    'GUATAPARA',
    'ITAPECERICA DA SERRA',
    'ITAPEVI',
    'ITAQUAQUECETUBA',
    'JABOTICABAL',
    'JANDIRA',
    'JARDINOPOLIS',
    'JULIO MESQUITA',
    'JUQUITIBA',
    'LUIS ANTONIO',
    'LUPERCIO',
    'MAIRIPORA',
    'MARILIA',
    'MAUA',
    'MOGI DAS CRUZES',
    'MONTE ALTO',
    'OCAUCU',
    'ORIENTE',
    'OSASCO',
    'OSCAR BRESSANE',
    'PIRAPORA DO BOM JESUS',
    'PITANGUEIRAS',
    'POA',
    'POMPEIA',
    'PONTAL',
    'PRADOPOLIS',
    'QUINTANA',
    'RIBEIRAO PIRES',
    'RIBEIRAO PRETO',
    'RIO GRANDE DA SERRA',
    'SALESOPOLIS',
    'SANTA CRUZ DA ESPERANCA',
    'SANTA ERNESTINA',
    'SANTA ISABEL',
    'SANTA ROSA DE VITERBO',
    'SANTANA DE PARNAIBA',
    'SANTO ANDRE',
    'SANTO ANTONIO DA ALEGRIA',
    'SAO BERNARDO DO CAMPO',
    'SAO CAETANO DO SUL',
    'SAO LOURENCO DA SERRA',
    'SAO PAULO',
    'SAO SIMAO',
    'SERRA AZUL',
    'SERRANA',
    'SERTAOZINHO',
    'SUZANO',
    'TABOAO DA SERRA',
    'VARGEM GRANDE PAULISTA',
    'VERA CRUZ',
    'ADRIANOPOLIS',
    'AGUDOS DO SUL',
    'ALMIRANTE TAMANDARE',
    'ANGULO',
    'ARAUCARIA',
    'ASTORGA',
    'ATALAIA',
    'BALSA NOVA',
    'BOCAIUVA DO SUL',
    'CAMPINA GRANDE DO SUL',
    'CAMPO DO TENENTE',
    'CAMPO LARGO',
    'CAMPO MAGRO',
    'CERRO AZUL',
    'COLOMBO',
    'CONTENDA',
    'CURITIBA',
    'DOUTOR CAMARGO',
    'DOUTOR ULYSSES',
    'FAZENDA RIO GRANDE',
    'FLORAI',
    'FLORESTA',
    'FLORIDA',
    'IGUARACU',
    'ITAMBE',
    'ITAPERUCU',
    'IVATUBA',
    'LAPA',
    'MANDAGUACU',
    'MANDAGUARI',
    'MANDIRITUBA',
    'MARIALVA',
    'MARINGA',
    'MUNHOZ DE MELO',
    'NOVA ESPERANCA',
    'OURIZONA',
    'PAICANDU',
    'PIEN',
    'PINHAIS',
    'PIRAQUARA',
    'PRESIDENTE CASTELO BRANCO',
    'QUATRO BARRAS',
    'QUITANDINHA',
    'RIO BRANCO DO SUL',
    'RIO NEGRO',
    'SANTA FE',
    'SAO JORGE DO IVAI',
    'SAO JOSE DOS PINHAIS',
    'SARANDI',
    'TIJUCAS DO SUL',
    'TUNAS DO PARANA',
    'UNIFLOR',
    'ABDON BATISTA',
    'AGUA DOCE',
    'ANCHIETA',
    'ARARANGUA',
    'BALNEARIO ARROIO DO SILVA',
    'BALNEARIO GAIVOTA',
    'BANDEIRANTE',
    'BARRA BONITA',
    'BELMONTE',
    'CAMPOS NOVOS',
    'CAPINZAL',
    'CATANDUVAS',
    'CELSO RAMOS',
    'DESCANSO',
    'DIONISIO CERQUEIRA',
    'ERMO',
    'ERVAL VELHO',
    'FLOR DO SERTAO',
    'GUARACIABA',
    'GUARUJA DO SUL',
    'HERVAL DOESTE',
    'IBICARE',
    'IPORA DO OESTE',
    'ITAPIRANGA',
    'JABORA',
    'JACINTO MACHADO',
    'JOACABA',
    'LACERDOPOLIS',
    'LUZERNA',
    'MARACAJA',
    'MELEIRO',
    'OURO',
    'PALMA SOLA',
    'PARAISO',
    'PASSO DE TORRES',
    'PRAIA GRANDE',
    'PRINCESA',
    'ROMELANDIA',
    'SANTA HELENA',
    'SANTA ROSA DO SUL',
    'SAO JOAO DO OESTE',
    'SAO JOAO DO SUL',
    'SAO JOSE DO CEDRO',
    'SAO MIGUEL DO OESTE',
    'SOMBRIO',
    'TIMBE DO SUL',
    'TREZE TILIAS',
    'TUNAPOLIS',
    'TURVO',
    'VARGEM',
    'VARGEM BONITA',
    'ZORTEA',
    'ACEGUA',
    'AGUDO',
    'AJURICABA',
    'ALEGRETE',
    'ALTO ALEGRE',
    'ALTO FELIZ',
    'ALVORADA',
    'AMETISTA DO SUL',
    'ANDRE DA ROCHA',
    'ANTONIO PRADO',
    'ARROIO DOS RATOS',
    'AUGUSTO PESTANA',
    'BAGE',
    'BARAO',
    'BARAO DO TRIUNFO',
    'BARRA DO QUARAI',
    'BARRA DO RIBEIRO',
    'BENTO GONCALVES',
    'BOA VISTA DAS MISSOES',
    'BOA VISTA DO CADEADO',
    'BOA VISTA DO INCRA',
    'BOA VISTA DO SUL',
    'BOZANO',
    'CACEQUI',
    'CACHOEIRINHA',
    'CAICARA',
    'CAMBARA DO SUL',
    'CAMPESTRE DA SERRA',
    'CAMPOS BORGES',
    'CANDELARIA',
    'CANDIOTA',
    'CANELA',
    'CANOAS',
    'CAPIVARI DO SUL',
    'CARAA',
    'CARLOS BARBOSA',
    'CATUIPE',
    'CAXIAS DO SUL',
    'CHIAPETTA',
    'CONDOR',
    'CORONEL BARROS',
    'CORONEL BICACO',
    'CORONEL PILAR',
    'COTIPORA',
    'COXILHA',
    'CRISTAL DO SUL',
    'CRUZ ALTA',
    'DILERMANDO DE AGUIAR',
    'DOIS LAJEADOS',
    'DOM PEDRITO',
    'DONA FRANCISCA',
    'ELDORADO DO SUL',
    'ENCRUZILHADA DO SUL',
    'ERNESTINA',
    'ERVAL SECO',
    'ESPUMOSO',
    'ESTEIO',
    'FAGUNDES VARELA',
    'FARROUPILHA',
    'FAXINAL DO SOTURNO',
    'FELIZ',
    'FLORES DA CUNHA',
    'FORMIGUEIRO',
    'FORTALEZA DOS VALOS',
    'FREDERICO WESTPHALEN',
    'GARIBALDI',
    'GLORINHA',
    'GRAMADO',
    'GRAMADO XAVIER',
    'GRAVATAI',
    'GUABIJU',
    'GUAIBA',
    'GUAPORE',
    'HERVEIRAS',
    'HULHA NEGRA',
    'IBIRUBA',
    'IJUI',
    'INHACORA',
    'IPE',
    'IRAI',
    'ITAARA',
    'IVORA',
    'JABOTICABA',
    'JACUIZINHO',
    'JAGUARI',
    'JARI',
    'JOIA',
    'JULIO DE CASTILHOS',
    'LAGOA DOS TRES CANTOS',
    'LIBERATO SALZANO',
    'MANOEL VIANA',
    'MARIANA PIMENTEL',
    'MATA',
    'MATO CASTELHANO',
    'MATO LEITAO',
    'MONTAURI',
    'MONTE BELO DO SUL',
    'MOSTARDAS',
    'NICOLAU VERGUEIRO',
    'NOVA ARACA',
    'NOVA BASSANO',
    'NOVA PADUA',
    'NOVA PALMA',
    'NOVA PETROPOLIS',
    'NOVA PRATA',
    'NOVA RAMADA',
    'NOVA ROMA DO SUL',
    'NOVA SANTA RITA',
    'NOVO TIRADENTES',
    'PALMARES DO SUL',
    'PALMITINHO',
    'PANAMBI',
    'PANTANO GRANDE',
    'PARAI',
    'PASSO DO SOBRADO',
    'PASSO FUNDO',
    'PEDRAS ALTAS',
    'PEJUCARA',
    'PICADA CAFE',
    'PINHAL',
    'PINHAL GRANDE',
    'PINHEIRINHO DO VALE',
    'PINHEIRO MACHADO',
    'PINTO BANDEIRA',
    'PONTAO',
    'PORTO ALEGRE',
    'PROTASIO ALVES',
    'QUARAI',
    'QUEVEDOS',
    'QUINZE DE NOVEMBRO',
    'RESTINGA SECA',
    'RIO PARDO',
    'RODEIO BONITO',
    'RONDA ALTA',
    'ROSARIO DO SUL',
    'SALTO DO JACUI',
    'SANTA BARBARA DO SUL',
    'SANTA CRUZ DO SUL',
    'SANTA MARIA',
    'SANTA TEREZA',
    'SANTANA DO LIVRAMENTO',
    'SANTO ANTONIO DA PATRULHA',
    'SANTO AUGUSTO',
    'SAO FRANCISCO DE ASSIS',
    'SAO FRANCISCO DE PAULA',
    'SAO JOAO DO POLESINE',
    'SAO JORGE',
    'SAO MARCOS',
    'SAO MARTINHO DA SERRA',
    'SAO PEDRO DO SUL',
    'SAO SEPE',
    'SAO VALENTIM DO SUL',
    'SAO VALERIO DO SUL',
    'SAO VICENTE DO SUL',
    'SAPUCAIA DO SUL',
    'SEBERI',
    'SELBACH',
    'SERAFINA CORREA',
    'SERTAO',
    'SERTAO SANTANA',
    'SILVEIRA MARTINS',
    'SINIMBU',
    'TAPERA',
    'TAQUARUCU DO SUL',
    'TAVARES',
    'TIO HUGO',
    'TOROPI',
    'TRES PALMEIRAS',
    'TUPANCIRETA',
    'UNIAO DA SERRA',
    'URUGUAIANA',
    'VALE DO SOL',
    'VALE REAL',
    'VALE VERDE',
    'VENANCIO AIRES',
    'VERA CRUZ',
    'VERANOPOLIS',
    'VIAMAO',
    'VICENTE DUTRA',
    'VICTOR GRAEFF',
    'VILA FLORES',
    'VISTA ALEGRE',
    'VISTA ALEGRE DO PRATA',
    'ANHANGUERA',
    'CAMPO ALEGRE DE GOIAS',
    'CATALAO',
    'CUMARI',
    'DAVINOPOLIS',
    'GOIANDIRA',
    'IPAMERI',
    'NOVA AURORA',
    'OUVIDOR',
    'TRES RANCHOS',
    'BRASILIA',
    'AUTAZES',
    'BORBA',
    'CAREIRO',
    'CAREIRO DA VARZEA',
    'IRANDUBA',
    'MANAQUIRI',
    'MANAUS',
    'NOVA OLINDA DO NORTE',
    'PRESIDENTE FIGUEIREDO',
    'RIO PRETO DA EVA',
    'ACARA',
    'ANANINDEUA',
    'BARCARENA',
    'BELEM',
    'BENEVIDES',
    'BUJARU',
    'COLARES',
    'CONCORDIA DO PARA',
    'MARITUBA',
    'SANTA BARBARA DO PARA',
    'SANTA IZABEL DO PARA',
    'SANTO ANTONIO DO TAUA',
    'SAO CAETANO DE ODIVELAS',
    'TOMEACU',
    'VIGIA',
    'ALTO SANTO',
    'ERERE',
    'IRACEMA',
    'JAGUARETAMA',
    'JAGUARIBARA',
    'JAGUARIBE',
    'LIMOEIRO DO NORTE',
    'MORADA NOVA',
    'PALHANO',
    'PEREIRO',
    'POTIRETAMA',
    'QUIXERE',
    'RUSSAS',
    'SAO JOAO DO JAGUARIBE',
    'TABULEIRO DO NORTE',
    'ALAGOA GRANDE',
    'ALAGOA NOVA',
    'ALCANTIL',
    'ALGODAO DE JANDAIRA',
    'AREIA',
    'AREIAL',
    'AROEIRAS',
    'ASSUNCAO',
    'BARRA DE SANTANA',
    'BARRA DE SAO MIGUEL',
    'BOA VISTA',
    'BOQUEIRAO',
    'CABACEIRAS',
    'CAMPINA GRANDE',
    'CARAUBAS',
    'CATURITE',
    'CUBATI',
    'ESPERANCA',
    'FAGUNDES',
    'GADO BRAVO',
    'GURJAO',
    'INGA',
    'ITATUBA',
    'JUAZEIRINHO',
    'JUNCO DO SERIDO',
    'LAGOA SECA',
    'MASSARANDUBA',
    'MATINHAS',
    'MONTADAS',
    'OLIVEDOS',
    'POCINHOS',
    'PUXINANA',
    'QUEIMADAS',
    'REMIGIO',
    'RIACHAO DO BACAMARTE',
    'RIACHO DE SANTO ANTONIO',
    'SANTA CECILIA',
    'SANTO ANDRE',
    'SAO DOMINGOS DO CARIRI',
    'SAO JOAO DO CARIRI',
    'SAO SEBASTIAO DE LAGOA DE ROCA',
    'SAO VICENTE DO SERIDO',
    'SERRA REDONDA',
    'SOLEDADE',
    'TAPEROA',
    'TENORIO',
    'UMBUZEIRO',
    'ABAIRA',
    'ARACATU',
    'BRUMADO',
    'CAMACARI',
    'CANDEIAS',
    'CATU',
    'CATURAMA',
    'DIAS DAVILA',
    'DOM BASILIO',
    'ERICO CARDOSO',
    'ITANAGRA',
    'JUSSIAPE',
    'LAURO DE FREITAS',
    'LIVRAMENTO DE NOSSA SENHORA',
    'MADRE DE DEUS',
    'MALHADA DE PEDRAS',
    'MATA DE SAO JOAO',
    'PARAMIRIM',
    'POJUCA',
    'RIO DE CONTAS',
    'RIO DO PIRES',
    'SALVADOR',
    'SANTO AMARO',
    'SAO FRANCISCO DO CONDE',
    'SAO SEBASTIAO DO PASSE',
    'SAUBARA',
    'SIMOES FILHO',
    'TERRA NOVA',
    'AGUA COMPRIDA',
    'ALPINOPOLIS',
    'ALVARENGA',
    'ANDRELANDIA',
    'ARACITABA',
    'ARAGUARI',
    'ARANTINA',
    'ARAPORA',
    'ATALEIA',
    'BELMIRO BRAGA',
    'BELO HORIZONTE',
    'BETIM',
    'BIAS FORTES',
    'BOCAINA DE MINAS',
    'BOM JARDIM DE MINAS',
    'BOM JESUS DA PENHA',
    'BOM JESUS DO GALHO',
    'BONFIM',
    'BRUMADINHO',
    'BURITIZEIRO',
    'CAETE',
    'CAMPANARIO',
    'CAMPINA VERDE',
    'CAMPO FLORIDO',
    'CANAPOLIS',
    'CAPETINGA',
    'CARAI',
    'CARATINGA',
    'CARLOS CHAGAS',
    'CARMO DO RIO CLARO',
    'CARMOPOLIS DE MINAS',
    'CASCALHO RICO',
    'CASSIA',
    'CATUJI',
    'CENTRAL DE MINAS',
    'CENTRALINA',
    'CHACARA',
    'CHIADOR',
    'CLARAVAL',
    'CONCEICAO DAS ALAGOAS',
    'CONFINS',
    'CONQUISTA',
    'CONTAGEM',
    'CORONEL PACHECO',
    'CORREGO NOVO',
    'CRUCILANDIA',
    'DELFINOPOLIS',
    'DELTA',
    'ENTRE FOLHAS',
    'ESMERALDAS',
    'EWBANK DA CAMARA',
    'FLORESTAL',
    'FORTALEZA DE MINAS',
    'FRANCISCOPOLIS',
    'FREI GASPAR',
    'GOIANA',
    'GUAPE',
    'IBIAI',
    'IBIRACI',
    'IBIRITE',
    'IGARAPE',
    'IMBE DE MINAS',
    'INDIANOPOLIS',
    'INHAPIM',
    'ITABIRINHA',
    'ITAGUARA',
    'ITAIPE',
    'ITAMBACURI',
    'ITAOBIM',
    'ITAU DE MINAS',
    'JABOTICATUBAS',
    'JUATUBA',
    'JUIZ DE FORA',
    'LADAINHA',
    'LAGOA SANTA',
    'LASSANCE',
    'LIBERDADE',
    'LIMA DUARTE',
    'MALACACHETA',
    'MANTENA',
    'MARIO CAMPOS',
    'MATEUS LEME',
    'MATIAS BARBOSA',
    'MENDES PIMENTEL',
    'MOEDA',
    'MONTE ALEGRE DE MINAS',
    'MONTE FORMOSO',
    'NANUQUE',
    'NOVA BELEM',
    'NOVA LIMA',
    'NOVA MODICA',
    'NOVA PONTE',
    'NOVA UNIAO',
    'NOVO CRUZEIRO',
    'NOVO ORIENTE DE MINAS',
    'OLARIA',
    'OLIVEIRA',
    'OLIVEIRA FORTES',
    'OURO VERDE DE MINAS')
    AND rc_session_index = 1 -- Contabizando apenas um acesso ao tema por sessão
GROUP BY 1,2,3,4
ORDER BY e_derived_tstamp_date

In [0]:
wb_access = _sqldf.toPandas()

In [0]:
now = datetime.now().strftime("%Y%m%d")
wb_access.to_csv(f'raw_wb_access_dengue_{now}.csv', index=False)